# TRAKE — Eval Tier3 + Qwen (Run 4/5/6) tren Colab GPU

Chay 3 tang + Qwen2-VL tren **Colab GPU**, khong can Mongo/UI. Can: **Qdrant collection `jina`** (ban zip tu `collections/jina`, Qdrant **1.19.0**, 921k diem) + **video goc** cua video_id lam GT.

**Truoc khi chay:** Runtime -> Change runtime type -> **T4 GPU**.


## 1. Kiem tra GPU

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
import torch
print("torch", torch.__version__, "| cuda", torch.cuda.is_available())
assert torch.cuda.is_available(), "Chua bat GPU! Runtime -> Change runtime type -> T4 GPU"


## 2. Lay code (nhanh feature/overral-trake-huy)

In [ ]:
%cd /content
![ -d aic2026 ] || git clone https://github.com/khoinguyen248/aic2026.git
%cd /content/aic2026
!git fetch -q origin && git checkout feature/overral-trake-huy && git pull -q


## 3. Cai phu thuoc (GIU torch CUDA co san cua Colab -- khong cai lai torch)

In [ ]:
!pip -q install flask==3.0.3 python-dotenv==1.0.1 numpy==1.26.4 scipy==1.15.1 \
  Pillow==11.3.0 einops==0.8.1 timm==0.4.12 opencv-python-headless \
  sentencepiece==0.2.1 transformers==4.55.1 open-clip-torch==2.32.0 torchscale==0.2.0 \
  protobuf==3.20.3 ftfy==6.3.1 qdrant-client==1.15.1 pymongo==4.14.0 deep-translator \
  accelerate bitsandbytes qwen-vl-utils
print("deps OK")


## 4. Qdrant 1.19.0 + restore collection `jina`

**Sua `JINA_ZIP_URL`** = link zip `jina` cua ban (host de wget, hoac Google Drive o duoi).
Zip phai chua `config.json`, `shard_key_mapping.json`, `version.info`, thu muc `0/` (noi dung cua `collections/jina`).

In [ ]:
import os, subprocess, glob, shutil, time
%cd /content
# 4a. binary Qdrant 1.19.0 (khop version tao data)
!wget -q https://github.com/qdrant/qdrant/releases/download/v1.19.0/qdrant-x86_64-unknown-linux-musl.tar.gz -O qdrant.tgz
!tar xzf qdrant.tgz
assert os.path.isfile("/content/qdrant"), "Khong tai duoc binary qdrant 1.19.0"

# 4b. lay zip jina
JINA_ZIP_URL = "PUT_YOUR_JINA_ZIP_URL_HERE"   # <-- SUA
if str(JINA_ZIP_URL).startswith("http"):
    !wget -q "$JINA_ZIP_URL" -O /content/jina.zip
    JINA_ZIP = "/content/jina.zip"
else:
    JINA_ZIP = JINA_ZIP_URL

# 4c. giai nen & dat dung cho: /content/qdrant_storage/collections/jina/
shutil.rmtree("/content/_jina_tmp", ignore_errors=True); os.makedirs("/content/_jina_tmp", exist_ok=True)
!unzip -q "$JINA_ZIP" -d /content/_jina_tmp
cfg = glob.glob("/content/_jina_tmp/**/config.json", recursive=True)
assert cfg, "Khong thay config.json trong zip"
src = os.path.dirname(cfg[0])
dst = "/content/qdrant_storage/collections/jina"
shutil.rmtree("/content/qdrant_storage", ignore_errors=True); os.makedirs(dst, exist_ok=True)
for name in os.listdir(src):
    shutil.move(os.path.join(src, name), os.path.join(dst, name))
print("collection files:", os.listdir(dst))


In [ ]:
# 4d. chay Qdrant nen
env = dict(os.environ); env["QDRANT__STORAGE__STORAGE_PATH"] = "/content/qdrant_storage"
qproc = subprocess.Popen(["/content/qdrant"], env=env,
                         stdout=open("/content/qdrant.log", "w"), stderr=subprocess.STDOUT)
time.sleep(12)
from qdrant_client import QdrantClient
c = QdrantClient(url="http://localhost:6333", check_compatibility=False)
print("collections:", [x.name for x in c.get_collections().collections])
print("jina count:", c.count("jina"))


> **(Tuy chon) Google Drive** neu zip tren Drive:
> ```python
> from google.colab import drive; drive.mount('/content/drive')
> # roi dat JINA_ZIP_URL = "/content/drive/MyDrive/jina.zip" o Cell 4b
> ```

## 5. Tai video goc (L25 / L26 / M02)

Mac dinh tai **L25**. O nay liet ke cac mp4 de biet video_id.

In [ ]:
%cd /content
VIDEO_ZIPS = [
    "https://aic-data.ledo.io.vn/Videos_L25_a.zip",
    # "https://aic-data.ledo.io.vn/Videos_L26_a.zip",
    # "https://aic-data.ledo.io.vn/Videos_M02.zip",
]
import glob
os.makedirs("/content/videos_raw", exist_ok=True)
for u in VIDEO_ZIPS:
    fn = "/content/" + u.split("/")[-1]
    !wget -q "$u" -O "$fn"
    !unzip -q -o "$fn" -d /content/videos_raw
mp4s = glob.glob("/content/videos_raw/**/*.mp4", recursive=True)
print(len(mp4s), "mp4:")
for p in sorted(mp4s)[:40]: print(" ", p)


## 6. Chon video_id + dua vao VIDEO_ROOT ten `<video_id>.mp4`

In [ ]:
import glob, shutil, os
VIDEO_ID = "L25_V025"   # <-- SUA: video_id co trong index jina & co mp4 o tren
os.makedirs("/content/videos", exist_ok=True)
key = VIDEO_ID.replace("_", "").replace("-", "").lower()
cand = [p for p in glob.glob("/content/videos_raw/**/*.mp4", recursive=True)
        if key in os.path.basename(p).replace("_", "").replace("-", "").lower()]
assert cand, f"Khong thay mp4 cho {VIDEO_ID}"
shutil.copy(cand[0], f"/content/videos/{VIDEO_ID}.mp4")
print("prepared:", f"/content/videos/{VIDEO_ID}.mp4", "<-", cand[0])


## 7. Tao GT visual-only (xem video -> dien frame_ids dung)

`frame_ids` = so thu tu khung hinh goc (0-based cv2), tang dan. `fps` = fps that.

In [ ]:
import json, os
gt = [{
    "id": f"{VIDEO_ID}_test",
    "events": [
        "su kien 1 -- mo ta canh",
        "su kien 2 -- mo ta canh",
        "su kien 3 -- mo ta canh",
    ],
    "language": True,
    "model": "jina",
    "max_event_gap_s": None,
    "gt": {"video_id": VIDEO_ID, "frame_ids": [100, 200, 300], "fps": 25}
}]
os.makedirs("/content/aic2026/backendAIC2025/app/eval/out", exist_ok=True)
p = "/content/aic2026/backendAIC2025/app/eval/queries_gt.json"
json.dump(gt, open(p, "w"), ensure_ascii=False, indent=2)
print("saved", p); print(open(p, encoding="utf-8").read())


## 8. Chay harness -- Run 2 / 4 / 5 / 6 (Qwen bat, GPU)

In [ ]:
%cd /content/aic2026/backendAIC2025
%env QDRANT_URL=http://localhost:6333
%env SEARCH_DEVICE=cuda
%env VIDEO_ROOT=/content/videos
%env KEYFRAMES_PATH=/content/keyframes
%env TRAKE_TIER3_ENABLED=true
%env TRAKE_QWEN_RERANK_ENABLED=true
%env QWEN_MODEL_PATH=Qwen/Qwen2-VL-2B-Instruct
%env QWEN_QUANTIZATION=4bit
%env QWEN_DEVICE_MAP=cuda
%env TRAKE_TOP_M=300
!python -m app.eval.eval_trake --queries app/eval/queries_gt.json \
   --out app/eval/out --runs run2_dp,run4_tier3_argmax,run5_tier3_peak,run6_full \
   --k 1,5,10,100 --tol-frames 25


In [ ]:
print(open("/content/aic2026/backendAIC2025/app/eval/out/eval_results.md", encoding="utf-8").read())


## Ghi chu
- Phai thay `tier3_globally_ready=True` va `qwen_globally_ready=True`. Neu False: kiem tra video `/content/videos/<video_id>.mp4` va GPU.
- **Latency o day la so cua Colab (T4).** Run1/Run3 (greedy, multimodal can Atlas) chay tren may Docker local; ghi ro hardware tung bang, dung Run2 lam moc quy doi.
- `VideoHit=0` (Tang 1 khong lay dung video): tang `TRAKE_TOP_M` (Cell 8) hoac viet mo ta su kien ro hon.
- Kendall/Spearman ~ 1 do DP ep don dieu.
